# Задание 4



## 📋 Описание задания
**Цель:** Классификация вин на 3 класса с подбором оптимальных гиперпараметров  
**Датасет:** Wine (178 образцов, 13 химических признаков)  
**Методы:** GridSearchCV для подбора C, penalty, solver  
**Концепции:** Подбор гиперпараметров, L1/L2 регуляризация


In [10]:
# ========================================================================
# ИМПОРТ БИБЛИОТЕК И ЗАГРУЗКА WINE DATASET
# ========================================================================
# Wine dataset: 178 образцов вина, 3 класса (сорта винограда)
# 13 признаков: алкоголь, яблочная кислота, зола, магний и т.д.
# ========================================================================

from sklearn.datasets import load_wine

wine = load_wine()

In [11]:
# ========================================================================
# РАЗДЕЛЕНИЕ НА TRAIN/TEST
# ========================================================================
# test_size=0.25 - 25% для тестирования
# random_state=42 - фиксируем разделение для воспроизводимости
# ========================================================================

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(wine.data, wine.target, test_size=0.25, random_state=42)

In [12]:
# ========================================================================
# СОЗДАНИЕ PIPELINE
# ========================================================================
# 1. StandardScaler - нормализация (важно для LogisticRegression!)
# 2. LogisticRegression(max_iter=200) - увеличен лимит итераций для сходимости
# ========================================================================

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

pipeline_wine = Pipeline([
    ('scaler', StandardScaler()),
    ('log_reg', LogisticRegression(max_iter = 200))
])

In [13]:
# ========================================================================
# СЕТКА ГИПЕРПАРАМЕТРОВ ДЛЯ ПОИСКА
# ========================================================================
# C - обратная сила регуляризации (от 0.01 до 100)
# penalty - тип регуляризации:
#   'l1' - Lasso (зануляет коэффициенты, отбор признаков)
#   'l2' - Ridge (уменьшает коэффициенты)
# solver - алгоритм оптимизации:
#   'multinomial' - одновременно обучает все классы
#   'saga' - быстрый алгоритм, поддерживает L1 и L2
# Всего комбинаций: 50 значений C × 2 penalty × 2 solver = 200 моделей!
# ========================================================================

import numpy as np

param_grid = {'log_reg__C': np.logspace(-2, 2),
             'log_reg__penalty': ['l1', 'l2'],
             'log_reg__solver': ['multinomial', 'saga']}

In [14]:
# ========================================================================
# СЕТКА ГИПЕРПАРАМЕТРОВ ДЛЯ ПОИСКА
# ========================================================================
# C - обратная сила регуляризации (от 0.01 до 100)
# penalty - тип регуляризации:
#   'l1' - Lasso (зануляет коэффициенты, отбор признаков)
#   'l2' - Ridge (уменьшает коэффициенты)
# solver - алгоритм оптимизации:
#   'multinomial' - одновременно обучает все классы
#   'saga' - быстрый алгоритм, поддерживает L1 и L2
# Всего комбинаций: 50 значений C × 2 penalty × 2 solver = 200 моделей!
# ========================================================================

from sklearn.model_selection import GridSearchCV
import warnings
warnings.filterwarnings("ignore")

grid_searcher = GridSearchCV(cv = 5, estimator = pipeline_wine, param_grid=param_grid, scoring = 'f1_weighted')

grid_searcher.fit(X_train, y_train)

,estimator,Pipeline(step...x_iter=200))])
,param_grid,"{'log_reg__C': array([1.0000...00000000e+02]), 'log_reg__penalty': ['l1', 'l2'], 'log_reg__solver': ['multinomial', 'saga']}"
,scoring,'f1_weighted'
,n_jobs,None
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,copy,True


In [15]:
grid_searcher.best_score_, grid_searcher.best_params_

(np.float64(0.9849771991567657),
 {'log_reg__C': np.float64(1.0985411419875584),
  'log_reg__penalty': 'l2',
  'log_reg__solver': 'saga'})

In [16]:
best_pipeline = grid_searcher.best_estimator_

In [17]:
from sklearn.metrics import classification_report

print(classification_report(y_test, best_pipeline.predict(X_test)))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        15
           1       1.00      0.94      0.97        18
           2       0.92      1.00      0.96        12

    accuracy                           0.98        45
   macro avg       0.97      0.98      0.98        45
weighted avg       0.98      0.98      0.98        45



In [18]:
import pickle as pkl
import os

# Создаём директорию, если её нет
os.makedirs('./saved models', exist_ok=True)

# Сохраняем модель
with open('./saved models/log_regression_4.pkl', 'wb') as file:
    pkl.dump(best_pipeline, file)

# Ответы на вопросы по логистической регрессии и настройке моделей

## 1. Что означает параметр C в логистической регрессии?

**Параметр C** представляет собой обратную силу регуляризации:
- Меньшие значения C → более сильная регуляризация
- Большие значения C → более слабая регуляризация
- C = 1.0 значение по умолчанию в scikit-learn
- C → ∞ соответствует отсутствию регуляризации

## 2. Как он связан с регуляризацией?

**Связь C с регуляризацией:**
- C = 1/λ, где λ - параметр регуляризации
- Высокие значения C минимизируют регуляризацию, позволяя модели подстраиваться под данные
- Низкие значения C усиливают регуляризацию, упрощая модель
- Регуляризация штрафует большие коэффициенты для предотвращения переобучения

## 3. Почему не все комбинации penalty и solver допустимы? Приведите пример недопустимой пары.

**Причины несовместимости:**
- Разные алгоритмы оптимизации поддерживают разные типы регуляризации
- Некоторые солверы не реализуют определенные penalty-функции

**Пример недопустимой пары:**
- `penalty='l1'` + `solver='lbfgs'` - недопустимо
- LBFGS не поддерживает L1-регуляризацию
- Правильная комбинация: `penalty='l1'` + `solver='liblinear'` или `'saga'`

## 4. Зачем увеличивать max_iter при использовании регуляризации?

**Причины увеличения max_iter:**
- Регуляризация замедляет сходимость алгоритма
- Сильная регуляризация требует больше итераций для достижения оптимума
- Жесткие ограничения на коэффициенты усложняют оптимизацию
- Предотвращение преждевременной остановки до сходимости

## 5. Почему в GridSearchCV используется метрика f1_weighted, а не accuracy?

**Преимущества f1_weighted над accuracy:**
- Учитывает дисбаланс классов в данных
- Балансирует precision и recall для каждого класса
- Более информативна при неравномерном распределении классов
- Менее чувствительна к доминирующему классу
- Обеспечивает более надежную оценку качества в многоклассовых задачах

## 6. Как влияет выбор регуляризации (l1 vs l2) на интерпретируемость модели?

**L1-регуляризация (Lasso):**
- Обнуляет коэффициенты неважных признаков
- Создает разреженные модели
- Упрощает интерпретацию за счет отбора признаков
- Позволяет идентифицировать наиболее значимые переменные

**L2-регуляризация (Ridge):**
- Уменьшает все коэффициенты равномерно
- Сохраняет все признаки в модели
- Коэффициенты отражают относительную важность признаков
- Менее интерпретируема при большом количестве признаков

## 7. Что происходит, если не указать solver, совместимый с penalty='l1'?

**Последствия несовместимой комбинации:**
- Scikit-learn выдает ValueError с сообщением о несовместимости
- Модель не может быть обучена
- Необходимо явно указать совместимый solver
- Для L1-регуляризации подходят: 'liblinear', 'saga'

## 8. Как кросс-валидация помогает избежать переобучения при подборе гиперпараметров?

**Механизм защиты от переобучения:**
- Оценка качества на разных подвыборках данных
- Исключение зависимости от конкретного разбиения
- Объективная оценка обобщающей способности
- Выбор гиперпараметров, работающих стабильно на разных данных
- Снижение variance оценки качества модели

## 9. Почему конвейер (pipeline) сохраняется целиком, а не только модель?

**Преимущества сохранения всего pipeline:**
- Гарантирует идентичность предобработки при развертывании
- Исключает рассинхронизацию шагов предобработки и модели
- Упрощает deployment - одна модель вместо нескольких компонентов
- Обеспечивает воспроизводимость всего workflow
- Сохраняет параметры трансформеров (например, средние и стандартные отклонения)
- Позволяет использовать модель на новых данных без дополнительной настройки